## ▶ What you'll see when you run this
- A generated 'PALOMAR CAFE' sign image, then **Claude and Gemini** reading it back as **alt-text** and exact **OCR** lines (`Open 7am - 3pm`, ...).

**Time:** ~10 min · **Cost:** free (cheapest model: Gemini Flash / Claude Haiku) · **Keys:** none to build/preview the image — add `ANTHROPIC_API_KEY` and/or `GEMINI_API_KEY` for the vision calls (each is skipped gracefully if missing).

# Week 15 · Notebook 1 — Multimodal Basics (Describe & OCR)
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Send the **same image + prompt** to **Claude** and **Gemini** vision models and compare. We make a self-contained sample image with Matplotlib so the notebook runs even before you add an image of your own.

> **Keys:** In Colab use the 🔑 *Secrets* panel and `userdata.get('NAME')`. Locally use environment variables. **Never commit keys.** If a key is missing, each cell prints a clear message instead of crashing.

## 0. Install SDKs

In [ ]:
!pip -q install anthropic google-generativeai pillow matplotlib

## 1. Load API keys safely (optional — cells degrade gracefully)

In [ ]:
import os
try:
    from google.colab import userdata  # Colab
    for k in ('ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except Exception:
    pass  # locally, set these in your shell environment
HAVE_CLAUDE = bool(os.environ.get('ANTHROPIC_API_KEY'))
HAVE_GEMINI = bool(os.environ.get('GEMINI_API_KEY'))
print('Claude key set:', HAVE_CLAUDE)
print('Gemini key set:', HAVE_GEMINI)

## 2. Make a self-contained sample image
We draw a simple 'storefront sign' with text so we can test both **description** and **OCR** without downloading anything.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9,
             facecolor='#0B1F3A', edgecolor='#128C8C', linewidth=4))
ax.text(0.5, 0.62, 'PALOMAR CAFE', ha='center', va='center',
        color='white', fontsize=26, fontweight='bold')
ax.text(0.5, 0.42, 'Open 7am - 3pm', ha='center', va='center',
        color='#CFE3E3', fontsize=16)
ax.text(0.5, 0.28, 'Coffee  *  Bagels  *  Wifi', ha='center', va='center',
        color='#CFE3E3', fontsize=14)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
fig.savefig('sign.png', dpi=100, bbox_inches='tight')
plt.close(fig)
print('wrote sign.png')

In [ ]:
from PIL import Image
Image.open('sign.png')   # preview the image inline

## 3. Helpers: send an image to Claude and to Gemini
Both helpers return a string and never crash if a key is missing.

In [ ]:
import base64, os

def media_type_for(path):
    """Map a file extension to the Claude media_type (students may upload JPG)."""
    ext = os.path.splitext(path)[1].lower()
    return {'.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
            '.png': 'image/png', '.gif': 'image/gif',
            '.webp': 'image/webp'}.get(ext, 'image/png')

def ask_claude(image_path, prompt, model='claude-sonnet-4-6'):
    if not HAVE_CLAUDE:
        return '[Claude skipped — no ANTHROPIC_API_KEY set]'
    import anthropic
    client = anthropic.Anthropic()
    b64 = base64.standard_b64encode(open(image_path, 'rb').read()).decode()
    msg = client.messages.create(
        model=model, max_tokens=500,
        messages=[{'role': 'user', 'content': [
            {'type': 'image', 'source': {'type': 'base64',
                'media_type': media_type_for(image_path), 'data': b64}},
            {'type': 'text', 'text': prompt}]}])
    return msg.content[0].text

In [ ]:
def ask_gemini(image_path, prompt, model='gemini-2.5-flash'):
    if not HAVE_GEMINI:
        return '[Gemini skipped — no GEMINI_API_KEY set]'
    import google.generativeai as genai
    from PIL import Image
    genai.configure(api_key=os.environ['GEMINI_API_KEY'])
    m = genai.GenerativeModel(model)
    resp = m.generate_content([prompt, Image.open(image_path)])
    return resp.text

## 4. Task A — describe the image

In [ ]:
PROMPT_DESC = 'In two sentences, describe this image as screen-reader alt-text.'
print('CLAUDE:\n', ask_claude('sign.png', PROMPT_DESC))
print('\nGEMINI:\n', ask_gemini('sign.png', PROMPT_DESC))

## 5. Task B — OCR (read the text exactly)

In [ ]:
PROMPT_OCR = ('Extract every piece of text in this image, exactly as written, '
              'one line per text element. Do not add commentary.')
print('CLAUDE:\n', ask_claude('sign.png', PROMPT_OCR))
print('\nGEMINI:\n', ask_gemini('sign.png', PROMPT_OCR))

## 6. Compare
Write ~150 words: did Claude and Gemini read the sign the same way? Which description was more useful as alt-text? Where did either add details that were not in the image (hallucination)? **Try your own photo:** upload an image, set `image_path`, and re-run. (This feeds your Final Project progress update.)

In [ ]:
# notes:
